# Prédiction des complications de santé à 6 mois

Ce notebook présente la démarche complète du projet : nettoyage des données,
exploration, puis modélisation. Le code de production correspondant se trouve
dans `src/` (`data_cleaning.py`, `train_model.py`, `predict.py`) — ce notebook
sert de **rapport lisible**, pas de code source à maintenir.


In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_cleaning import clean_dataset, TARGET_COL

pd.set_option("display.max_columns", 30)


## 1. Nettoyage des données

Le dataset brut contient des valeurs très hétérogènes : espaces parasites dans
les colonnes, virgules décimales, unités mélangées au texte (`mg/L`),
préfixes/suffixes (`age:`, `ans`), catégories de sexe non standardisées (`M`,
`Femme`, `h`...), et des incohérences physiologiques (tension systolique ≤
diastolique). Toute la logique est dans `src/data_cleaning.py`.


In [ ]:
df = clean_dataset("../data/raw/projet3_sante_complications.csv")
print(df.shape)
df.head()


In [ ]:
df.isnull().sum().sum(), df.duplicated().sum()


## 2. Exploration rapide

La cible `complication_6m` est déséquilibrée (~79% de complications contre
~21%), ce qui justifie l'usage de SMOTE en modélisation (section 3).


In [ ]:
df[TARGET_COL].value_counts(normalize=True) * 100


In [ ]:
fig, ax = plt.subplots(figsize=(5,4))
df[TARGET_COL].value_counts().sort_index().plot(kind="bar", ax=ax, color=["#4C72B0","#DD8452"])
ax.set_xticklabels(["Pas de complication (0)", "Complication (1)"], rotation=0)
ax.set_title("Répartition de la variable cible")
plt.tight_layout()
plt.show()


In [ ]:
num_df = df.select_dtypes(include=np.number)
corr = num_df.corr()
fig, ax = plt.subplots(figsize=(10,8))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns))); ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=90, fontsize=7)
ax.set_yticklabels(corr.columns, fontsize=7)
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Matrice de corrélation")
plt.tight_layout()
plt.show()


## 3. Modélisation

### 3.1 Préparation

On sépare X/y, on encode `sexe`, puis on effectue un split stratifié
train/test (80/20, `random_state=42` pour la reproductibilité).


In [ ]:
from sklearn.model_selection import train_test_split

y = df[TARGET_COL]
X = df.drop(columns=[TARGET_COL])
X = pd.get_dummies(X, columns=["sexe"], drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train.shape, X_test.shape


### 3.2 Round 1 — modèles de référence (sans gestion du déséquilibre)

Trois modèles sont comparés : Logistic Regression, Random Forest, Gradient
Boosting. Chacun est encapsulé dans un `Pipeline` scikit-learn avec
imputation (médiane apprise **sur le train uniquement**, contrairement à la
version initiale du notebook qui ré-imputait le test avec sa propre médiane).


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

baseline_models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000)),
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(n_estimators=300, random_state=42)),
    ]),
    "Gradient Boosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", GradientBoostingClassifier(random_state=42)),
    ]),
}

baseline_results = {}
for name, pipe in baseline_models.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    baseline_results[name] = {
        "accuracy": accuracy_score(y_test, y_pred),
        "f1_macro": f1_score(y_test, y_pred, average="macro"),
    }

pd.DataFrame(baseline_results).T


### 3.3 Round 2 — SMOTE (retenu comme version finale)

Le déséquilibre de classes (~79/21) pénalise le rappel sur la classe
minoritaire (patients qui *ne* développent *pas* de complication) : le
Random Forest baseline obtient par exemple un excellent accuracy mais un
rappel très faible sur la classe 0. On applique donc SMOTE (uniquement sur
les plis d'entraînement, via `imblearn.pipeline.Pipeline`) pour rééquilibrer
les classes avant l'entraînement.

**C'est cette version qui est utilisée par `src/train_model.py`**
et qui correspond au modèle livré (`models/modele_final.pkl`).


In [ ]:
try:
    from imblearn.pipeline import Pipeline as ImbPipeline
    from imblearn.over_sampling import SMOTE
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False
    print("imbalanced-learn n'est pas installé dans cet environnement : "
          "voir requirements.txt (pip install imbalanced-learn) pour reproduire "
          "cette section avec SMOTE. src/train_model.py bascule automatiquement "
          "sur class_weight='balanced' si SMOTE est indisponible.")

if HAS_IMBLEARN:
    smote_models = {
        "Logistic Regression": ImbPipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("smote", SMOTE(random_state=42)),
            ("model", LogisticRegression(max_iter=1000)),
        ]),
        "Random Forest": ImbPipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("smote", SMOTE(random_state=42)),
            ("model", RandomForestClassifier(n_estimators=300, random_state=42)),
        ]),
        "Gradient Boosting": ImbPipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("smote", SMOTE(random_state=42)),
            ("model", GradientBoostingClassifier(random_state=42)),
        ]),
    }

    smote_results = {}
    fitted_smote = {}
    for name, pipe in smote_models.items():
        pipe.fit(X_train, y_train)
        fitted_smote[name] = pipe
        y_pred = pipe.predict(X_test)
        smote_results[name] = {
            "accuracy": accuracy_score(y_test, y_pred),
            "f1_macro": f1_score(y_test, y_pred, average="macro"),
        }
    display(pd.DataFrame(smote_results).T)


### 3.4 Comparaison et choix final

Le modèle retenu est celui qui maximise le **F1 macro** (et non l'accuracy
seule), car l'accuracy est trompeuse sur un jeu déséquilibré : un modèle qui
prédit presque toujours la classe majoritaire obtient une bonne accuracy
mais un rappel nul sur la classe minoritaire. C'est exactement ce qui a été
observé sur le Random Forest baseline (round 1) : accuracy élevée, rappel
proche de 0 sur la classe 0. Cf. `models/metrics.json` produit par
`src/train_model.py` pour les métriques détaillées du modèle livré.


## 4. Sauvegarde des artefacts

Gérée par `src/train_model.py` :
- `models/modele_final.pkl` — pipeline complet (imputation + [SMOTE] + modèle)
- `models/feature_columns.pkl` — ordre des colonnes attendu par le modèle
- `models/metrics.json` — métriques du modèle retenu

Contrairement à la version initiale du projet, **aucun `scaler.pkl` externe
n'est nécessaire** : le scaling (quand il est utile, i.e. pour la régression
logistique) est intégré au pipeline sauvegardé. L'ancien `scaler.pkl` était
d'ailleurs orphelin : il était ajusté sur les données du round 1 (baseline)
mais le modèle final livré (`modele_final.pkl`, round 2 avec SMOTE) ne
l'utilisait pas du tout.
